# 05 revenue validation

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive", force_remount=False)

# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

REVENUE_FILE = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_REVENUE_DATA.csv"
)

print("=" * 60)
print("NOTEBOOK 05 — GOOGLE DRIVE / PROJECT CHECK")
print("=" * 60)

print("Drive path exists:", os.path.exists("/content/drive"))
print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))
print("Project exists:", os.path.exists(PROJECT_ROOT))

print("\nRevenue dataset:")
print(REVENUE_FILE)
print("File exists:", os.path.exists(REVENUE_FILE))

In [ ]:
import pandas as pd

# ============================================================
# LOAD FINAL REVENUE DATASET
# ============================================================

revenue_df = pd.read_csv(REVENUE_FILE)

print("=" * 60)
print("NOTEBOOK 05 — REVENUE DATASET LOAD")
print("=" * 60)

print("Dataset loaded successfully.")
print("Rows:", len(revenue_df))
print("Columns:", len(revenue_df.columns))

print("\nColumn names:")
for i, col in enumerate(revenue_df.columns, start=1):
    print(f"{i}. {col}")

print("\nData types:")
print(revenue_df.dtypes)

print("\nFirst 5 records:")
display(revenue_df.head())

In [ ]:
# ============================================================
# RECORD INTEGRITY VALIDATION
# ============================================================

print("=" * 60)
print("NOTEBOOK 05 — RECORD INTEGRITY VALIDATION")
print("=" * 60)

# Expected number of records
expected_records = 130

# Basic counts
unique_movie_records = revenue_df["movie_record_id"].nunique()
unique_tmdb_ids = revenue_df["tmdb_id"].nunique()

missing_movie_records = revenue_df["movie_record_id"].isna().sum()
missing_tmdb_ids = revenue_df["tmdb_id"].isna().sum()

duplicate_movie_records = revenue_df["movie_record_id"].duplicated().sum()
duplicate_tmdb_ids = revenue_df["tmdb_id"].duplicated().sum()

print("Expected records:", expected_records)
print("Actual records:", len(revenue_df))

print("\nUnique movie_record_id:", unique_movie_records)
print("Unique tmdb_id:", unique_tmdb_ids)

print("\nMissing movie_record_id:", missing_movie_records)
print("Missing tmdb_id:", missing_tmdb_ids)

print("\nDuplicate movie_record_id:", duplicate_movie_records)
print("Duplicate tmdb_id:", duplicate_tmdb_ids)

# Final validation
integrity_pass = (
    len(revenue_df) == expected_records
    and unique_movie_records == expected_records
    and unique_tmdb_ids == expected_records
    and missing_movie_records == 0
    and missing_tmdb_ids == 0
    and duplicate_movie_records == 0
    and duplicate_tmdb_ids == 0
)

print("\n" + "=" * 60)

if integrity_pass:
    print("✅ RECORD INTEGRITY VALIDATION PASSED")
else:
    print("❌ RECORD INTEGRITY VALIDATION FAILED")

In [ ]:
# ============================================================
# YEAR / RANK VALIDATION
# ============================================================

print("=" * 60)
print("NOTEBOOK 05 — YEAR / RANK VALIDATION")
print("=" * 60)

expected_years = list(range(2000, 2026))

# ------------------------------------------------------------
# Year range
# ------------------------------------------------------------

actual_years = sorted(revenue_df["year"].unique())

missing_years = sorted(set(expected_years) - set(actual_years))
extra_years = sorted(set(actual_years) - set(expected_years))

print("Expected years:", expected_years[0], "to", expected_years[-1])
print("Actual years:", actual_years[0], "to", actual_years[-1])

print("\nMissing years:", missing_years)
print("Extra years:", extra_years)

# ------------------------------------------------------------
# Movies per year
# ------------------------------------------------------------

movies_per_year = revenue_df.groupby("year").size()

print("\nMovies per year:")
print(movies_per_year.to_string())

invalid_year_counts = movies_per_year[movies_per_year != 5]

print("\nYears not having exactly 5 movies:")
print(invalid_year_counts if len(invalid_year_counts) > 0 else "None")

# ------------------------------------------------------------
# Rank validation
# ------------------------------------------------------------

invalid_rank_years = []

for year in expected_years:
    year_data = revenue_df[revenue_df["year"] == year]

    ranks = sorted(year_data["rank"].tolist())

    if ranks != [1, 2, 3, 4, 5]:
        invalid_rank_years.append({
            "year": year,
            "ranks": ranks
        })

print("\nYears with invalid ranks:")
print(invalid_rank_years if invalid_rank_years else "None")

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

year_rank_pass = (
    actual_years == expected_years
    and len(invalid_year_counts) == 0
    and len(invalid_rank_years) == 0
)

print("\n" + "=" * 60)

if year_rank_pass:
    print("✅ YEAR / RANK VALIDATION PASSED")
else:
    print("❌ YEAR / RANK VALIDATION FAILED")

In [ ]:
# ============================================================
# INDIA GROSS VALIDATION
# ============================================================

print("=" * 60)
print("NOTEBOOK 05 — INDIA GROSS VALIDATION")
print("=" * 60)

gross_col = "india_gross_inr_cr"

missing_gross = revenue_df[gross_col].isna().sum()
zero_gross = (revenue_df[gross_col] == 0).sum()
negative_gross = (revenue_df[gross_col] < 0).sum()

print("Total records:", len(revenue_df))
print("Missing India Gross:", missing_gross)
print("Zero India Gross:", zero_gross)
print("Negative India Gross:", negative_gross)

print("\nMinimum India Gross:", revenue_df[gross_col].min())
print("Maximum India Gross:", revenue_df[gross_col].max())

print("\nData type:", revenue_df[gross_col].dtype)

# Check numeric type
is_numeric = pd.api.types.is_numeric_dtype(revenue_df[gross_col])

print("\nIndia Gross numeric:", is_numeric)

# Final validation
india_gross_pass = (
    missing_gross == 0
    and negative_gross == 0
    and is_numeric
)

print("\n" + "=" * 60)

if india_gross_pass:
    print("✅ INDIA GROSS VALIDATION PASSED")
else:
    print("❌ INDIA GROSS VALIDATION FAILED")

In [ ]:
# ============================================================
# TMDB FINANCIAL VALIDATION
# ============================================================

print("=" * 60)
print("TMDB FINANCIAL VALIDATION")
print("=" * 60)

budget_col = "tmdb_budget_usd"
revenue_col = "tmdb_worldwide_revenue_usd"

# Missing values
missing_budget = revenue_df[budget_col].isna().sum()
missing_revenue = revenue_df[revenue_col].isna().sum()

# Zero values
zero_budget = (revenue_df[budget_col] == 0).sum()
zero_revenue = (revenue_df[revenue_col] == 0).sum()

# Negative values
negative_budget = (revenue_df[budget_col] < 0).sum()
negative_revenue = (revenue_df[revenue_col] < 0).sum()

# Numeric checks
budget_numeric = pd.api.types.is_numeric_dtype(revenue_df[budget_col])
revenue_numeric = pd.api.types.is_numeric_dtype(revenue_df[revenue_col])

print("TMDB Budget:")
print("  Missing:", missing_budget)
print("  Zero:", zero_budget)
print("  Negative:", negative_budget)
print("  Numeric:", budget_numeric)

print("\nTMDB Worldwide Revenue:")
print("  Missing:", missing_revenue)
print("  Zero:", zero_revenue)
print("  Negative:", negative_revenue)
print("  Numeric:", revenue_numeric)

print("\nBudget minimum:", revenue_df[budget_col].min())
print("Budget maximum:", revenue_df[budget_col].max())

print("\nWorldwide revenue minimum:", revenue_df[revenue_col].min())
print("Worldwide revenue maximum:", revenue_df[revenue_col].max())

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

tmdb_financial_pass = (
    missing_budget == 0
    and missing_revenue == 0
    and negative_budget == 0
    and negative_revenue == 0
    and budget_numeric
    and revenue_numeric
)

print("\n" + "=" * 60)

if tmdb_financial_pass:
    print("✅ TMDB FINANCIAL VALIDATION PASSED")
else:
    print("❌ TMDB FINANCIAL VALIDATION FAILED")

In [ ]:
# ============================================================
# NOTEBOOK 05 — TMDB MAPPING CONSISTENCY VALIDATION
# ============================================================

print("=" * 60)
print("NOTEBOOK 05 — TMDB MAPPING CONSISTENCY VALIDATION")
print("=" * 60)

# Missing values
missing_tmdb_id = revenue_df["tmdb_id"].isna().sum()
missing_tmdb_title = revenue_df["tmdb_title"].isna().sum()

# Unique values
unique_tmdb_ids = revenue_df["tmdb_id"].nunique()
unique_tmdb_titles = revenue_df["tmdb_title"].nunique()

# Duplicate IDs
duplicate_tmdb_ids = revenue_df["tmdb_id"].duplicated().sum()

print("Total records:", len(revenue_df))

print("\nTMDB ID:")
print("  Missing:", missing_tmdb_id)
print("  Unique:", unique_tmdb_ids)
print("  Duplicates:", duplicate_tmdb_ids)

print("\nTMDB Title:")
print("  Missing:", missing_tmdb_title)
print("  Unique:", unique_tmdb_titles)

# ------------------------------------------------------------
# Check ID → title consistency
# ------------------------------------------------------------

id_title_counts = (
    revenue_df.groupby("tmdb_id")["tmdb_title"]
    .nunique()
)

inconsistent_id_title = id_title_counts[id_title_counts > 1]

print("\nTMDB IDs mapped to multiple titles:",
      len(inconsistent_id_title))

if len(inconsistent_id_title) > 0:
    print(inconsistent_id_title)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

tmdb_mapping_pass = (
    missing_tmdb_id == 0
    and missing_tmdb_title == 0
    and unique_tmdb_ids == 130
    and duplicate_tmdb_ids == 0
    and len(inconsistent_id_title) == 0
)

print("\n" + "=" * 60)

if tmdb_mapping_pass:
    print("✅ TMDB MAPPING CONSISTENCY VALIDATION PASSED")
else:
    print("❌ TMDB MAPPING CONSISTENCY VALIDATION FAILED")

In [ ]:
# ============================================================
# NOTEBOOK 05 — FINAL SCHEMA VALIDATION
# ============================================================

print("=" * 60)
print("NOTEBOOK 05 — FINAL SCHEMA VALIDATION")
print("=" * 60)

expected_columns = [
    "movie_record_id",
    "year",
    "rank",
    "movie_title",
    "language",
    "india_gross_inr_cr",
    "tmdb_id",
    "tmdb_title",
    "tmdb_budget_usd",
    "tmdb_worldwide_revenue_usd"
]

actual_columns = revenue_df.columns.tolist()

missing_columns = sorted(
    set(expected_columns) - set(actual_columns)
)

extra_columns = sorted(
    set(actual_columns) - set(expected_columns)
)

wrong_order = actual_columns != expected_columns

print("Expected columns:", len(expected_columns))
print("Actual columns:", len(actual_columns))

print("\nMissing columns:")
print(missing_columns if missing_columns else "None")

print("\nExtra columns:")
print(extra_columns if extra_columns else "None")

print("\nColumn order correct:", not wrong_order)

if wrong_order:
    print("\nActual column order:")
    for i, col in enumerate(actual_columns, start=1):
        print(f"{i}. {col}")

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

schema_pass = (
    actual_columns == expected_columns
)

print("\n" + "=" * 60)

if schema_pass:
    print("✅ FINAL SCHEMA VALIDATION PASSED")
else:
    print("❌ FINAL SCHEMA VALIDATION FAILED")

In [ ]:
# ============================================================
# NOTEBOOK 05 — FINAL VALIDATION SUMMARY
# ============================================================

print("=" * 60)
print("NOTEBOOK 05 — FINAL REVENUE DATA VALIDATION SUMMARY")
print("=" * 60)

validation_results = {
    "Record integrity": integrity_pass,
    "Year / rank validation": year_rank_pass,
    "India Gross validation": india_gross_pass,
    "TMDB financial validation": tmdb_financial_pass,
    "TMDB mapping consistency": tmdb_mapping_pass,
    "Final schema validation": schema_pass
}

for check, result in validation_results.items():
    print(f"{'✅ PASS' if result else '❌ FAIL'} — {check}")

overall_pass = all(validation_results.values())

print("\n" + "=" * 60)

if overall_pass:
    print("✅ NOTEBOOK 05 — ALL VALIDATIONS PASSED")
    print("✅ ICMB_130_REVENUE_DATA.csv IS VALIDATED")
    print("✅ REVENUE DATASET READY FOR NEXT STAGE")
else:
    print("❌ NOTEBOOK 05 — VALIDATION FAILED")
    print("⚠️ DO NOT PROCEED UNTIL THE FAILED CHECK IS RESOLVED")

print("=" * 60)